# Patient Segmentation using K-Means and Hierarchical Clustering
* Yusuf Imantaka Bastari
* Muhammad Javier
* Khrisna Dwi Haryanto


## Data Cleaning

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('patient_segmentation_dataset.csv')
pd.set_option('display.max_columns', None)

In [3]:
df.head()

,PatientID,Age,Gender,State,City,Height_cm,Weight_kg,BMI,Insurance_Type,Primary_Condition,Num_Chronic_Conditions,Annual_Visits,Avg_Billing_Amount,Last_Visit_Date,Days_Since_Last_Visit,Preventive_Care_Flag
0,P10000,64,Male,GA,Unknown,151,115,50.4,Private,Arthritis,3,7,2995.0,2025-07-18,186,0
1,P10001,59,Male,OH,Unknown,189,68,19.0,Medicare,Depression,1,8,1209.0,2025-12-12,39,0
2,P10002,58,Female,PA,Unknown,156,91,37.4,Private,Asthma,1,4,999.0,2025-09-16,126,0
3,P10003,43,Female,GA,Unknown,152,92,39.8,Medicare,Hypertension,1,6,5638.5,2025-04-09,286,1
4,P10004,53,Female,NC,Unknown,167,51,18.3,Medicaid,Asthma,1,4,5796.0,2025-03-07,319,0


In [4]:
df.dtypes

PatientID                     str
Age                         int64
Gender                        str
State                         str
City                          str
Height_cm                   int64
Weight_kg                   int64
BMI                       float64
Insurance_Type                str
Primary_Condition             str
Num_Chronic_Conditions      int64
Annual_Visits               int64
Avg_Billing_Amount        float64
Last_Visit_Date               str
Days_Since_Last_Visit       int64
Preventive_Care_Flag        int64
dtype: object

In [5]:
# Drop Unnecesary Columns

df_new = df.drop(columns=['PatientID'])

# Encode Categorical Variables

def encode_categorical(df):
    df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})
    df['State'] = df['State'].map(df['State'].value_counts(normalize=True).to_dict())
    df['City'] = df['City'].map(df['City'].value_counts(normalize=True).to_dict())
    df['Insurance_Type'] = df['Insurance_Type'].map(df['Insurance_Type'].value_counts(normalize=True).to_dict())
    df['Primary_Condition'] = df['Primary_Condition'].map(df['Primary_Condition'].value_counts(normalize=True).to_dict())

    return df

encoded_df = encode_categorical(df_new)

In [6]:
encoded_df.head()

,Age,Gender,State,City,Height_cm,Weight_kg,BMI,Insurance_Type,Primary_Condition,Num_Chronic_Conditions,Annual_Visits,Avg_Billing_Amount,Last_Visit_Date,Days_Since_Last_Visit,Preventive_Care_Flag
0,64,0,0.1055,0.506,151,115,50.4,0.2725,0.108306,3,7,2995.0,2025-07-18,186,0
1,59,0,0.1040,0.506,189,68,19.0,0.4530,0.106312,1,8,1209.0,2025-12-12,39,0
2,58,1,0.0945,0.506,156,91,37.4,0.2725,0.106312,1,4,999.0,2025-09-16,126,0
3,43,1,0.1055,0.506,152,92,39.8,0.4530,0.139535,1,6,5638.5,2025-04-09,286,1
4,53,1,0.1065,0.506,167,51,18.3,0.2415,0.106312,1,4,5796.0,2025-03-07,319,0


In [7]:
# Fix Date Time Column

encoded_df['Last_Visit_Date'] = pd.to_datetime(encoded_df['Last_Visit_Date'])

# Extract features from date time column

encoded_df['Last_Visit_Year'] = encoded_df['Last_Visit_Date'].dt.year
encoded_df['Last_Visit_Month'] = encoded_df['Last_Visit_Date'].dt.month
encoded_df['Last_Visit_Day'] = encoded_df['Last_Visit_Date'].dt.day

encoded_df = encoded_df.drop(columns=['Last_Visit_Date'])

In [8]:
encoded_df.head()

,Age,Gender,State,City,Height_cm,Weight_kg,BMI,Insurance_Type,Primary_Condition,Num_Chronic_Conditions,Annual_Visits,Avg_Billing_Amount,Days_Since_Last_Visit,Preventive_Care_Flag,Last_Visit_Year,Last_Visit_Month,Last_Visit_Day
0,64,0,0.1055,0.506,151,115,50.4,0.2725,0.108306,3,7,2995.0,186,0,2025,7,18
1,59,0,0.1040,0.506,189,68,19.0,0.4530,0.106312,1,8,1209.0,39,0,2025,12,12
2,58,1,0.0945,0.506,156,91,37.4,0.2725,0.106312,1,4,999.0,126,0,2025,9,16
3,43,1,0.1055,0.506,152,92,39.8,0.4530,0.139535,1,6,5638.5,286,1,2025,4,9
4,53,1,0.1065,0.506,167,51,18.3,0.2415,0.106312,1,4,5796.0,319,0,2025,3,7


# K-Means Clustering